# Taxonomy Analysis

Exploration de la ressource benchmark (`benchmark_resource.parquet`) :
évolution temporelle et répartition par taxonomie Mistral (feuille `task` ou mère `mother_task`).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
DATA_DIR = Path("../data")
RESOURCE_PATH = DATA_DIR / "corpus" / "single_task_benchmark_paper.parquet"

# "task" (feuille) ou "mother_task" (catégorie mère)
# TASK_COL = "task"  # "mother_task
TASK_COL = "mother_task"

benchmarks = pd.read_parquet(RESOURCE_PATH)
benchmarks = benchmarks[benchmarks["year"] < 2026].dropna(subset=[TASK_COL])

anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")
anthology = anthology[anthology["year"] < 2026]

print(f"{len(benchmarks):,} papers | grouping by: {TASK_COL}")
print(benchmarks[TASK_COL].nunique(), "distinct labels")


## FOS

In [ ]:
fos_hierarchy = pd.read_parquet(DATA_DIR / "fos_hierarchy" / "fos_hierarchy.parquet")
fos_hierarchy = fos_hierarchy[fos_hierarchy["types"] == "Task"]

In [ ]:
fos_hierarchy["types"].unique()

## Évolution temporelle du nombre de benchmarks

In [ ]:
year_counts = benchmarks.groupby("year").size()

g = sns.barplot(x=year_counts.index, y=year_counts.values)
g.set(
    xlabel="Year", ylabel="Number of benchmarks", title="Benchmarks introduced per year"
)
g.bar_label(g.containers[0])
g.figure.set_size_inches(8, 4)
g.figure.tight_layout()


In [ ]:
# Évolution comparée : benchmarks vs total de papiers dans l'anthology
bench_per_year = benchmarks.groupby("year").size().rename("benchmarks")
anthology_per_year = anthology.groupby("year").size().rename("anthology")

compared = pd.concat([anthology_per_year, bench_per_year], axis=1).fillna(0).astype(int)
compared["pct"] = (compared["benchmarks"] / compared["anthology"] * 100).round(2)

fig, ax1 = sns.mpl.pyplot.subplots(figsize=(10, 5))

sns.barplot(
    x=compared.index,
    y=compared["anthology"],
    color="lightgrey",
    label="All papers (anthology)",
    ax=ax1,
)
ax1.set_ylabel("Total papers (anthology)")
ax1.set_xlabel("Year")

ax2 = ax1.twinx()
sns.lineplot(
    x=range(len(compared)),
    y=compared["benchmarks"].values,
    color="tab:red",
    marker="o",
    label="Benchmarks (resource cohort)",
    ax=ax2,
)
ax2.set_ylabel("Benchmarks")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
ax2.get_legend().remove()

ax1.set_title("Benchmark cohort vs total ACL Anthology papers")
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()


In [ ]:
cum_bench = compared["benchmarks"].cumsum()
cum_anthology = compared["anthology"].cumsum()
compared["bench_share_cum"] = cum_bench / cum_anthology * 100

g = sns.lineplot(
    x=compared.index, y=compared["bench_share_cum"], marker="o", color="tab:blue"
)
g.set(
    xlabel="Year",
    ylabel="Cumulative % of anthology",
    title="Benchmark cohort as % of ACL Anthology papers (cumulative)",
)
g.axhline(
    compared["bench_share_cum"].iloc[-1],
    ls="--",
    color="grey",
    alpha=0.5,
    label=f"Final avg: {compared['bench_share_cum'].iloc[-1]:.2f}%",
)
g.legend()
g.figure.set_size_inches(9, 4)
g.figure.tight_layout()


## Task taxonomy (`TASK_COL`)


In [ ]:
task_counts = benchmarks[TASK_COL].value_counts()

g = sns.barplot(y=task_counts.index[:15], x=task_counts.values[:15], orient="h")
g.set(
    xlabel="Count",
    ylabel="",
    title=f"Top 15 labels ({TASK_COL})",
)
g.bar_label(g.containers[0])
g.figure.set_size_inches(8, 6)
g.figure.tight_layout()


## Task taxonomy × Year


In [ ]:
top_tasks = benchmarks[TASK_COL].value_counts().head(10).index
subset = benchmarks[benchmarks[TASK_COL].isin(top_tasks)]

pivot = subset.groupby([TASK_COL, "year"]).size().unstack(fill_value=0)
pivot = pivot.loc[top_tasks]

g = sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd")
g.set(xlabel="Year", ylabel="", title=f"Top 10 {TASK_COL} × Year")
g.figure.set_size_inches(8, 6)
g.figure.tight_layout()


In [ ]:
top_n = 10
top_tasks = benchmarks[TASK_COL].value_counts().head(top_n).index
subset = benchmarks[benchmarks[TASK_COL].isin(top_tasks)]

bubble = subset.groupby([TASK_COL, "year"]).size().reset_index(name="count")

task_order = list(top_tasks)
bubble["y"] = bubble[TASK_COL].map({t: i for i, t in enumerate(task_order)})

fig, ax = sns.mpl.pyplot.subplots(figsize=(14, 6))

max_count = bubble["count"].max()
sizes = (bubble["count"] / max_count) * 800

ax.scatter(
    bubble["year"],
    bubble["y"],
    s=sizes,
    alpha=0.6,
    color="steelblue",
    edgecolors="white",
    linewidth=0.5,
)

for _, row in bubble.iterrows():
    ax.annotate(
        str(int(row["count"])),
        (row["year"], row["y"]),
        ha="center",
        va="center",
        fontsize=8,
        color="black",
    )

ax.set_yticks(range(len(task_order)))
ax.set_yticklabels(task_order)
ax.invert_yaxis()

years = sorted(bubble["year"].unique())
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha="right")

ax.set_xlabel("Year")
ax.set_ylabel("")
ax.set_title(f"Top 10 {TASK_COL} over time (bubble size = count)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
